# Run once, reuse, and reopen a football experiment

## Goal and setup

This synthetic example prepares 30 fixtures, fits a fresh scaler and Ridge model,
reuses the completed result, and reopens its saved inputs and reports. It uses the
existing Python (misc314) kernel. No real football performance claim is made.

### 1. Prepare inspectable inputs and the held-out scope

Rows 0–19 train the model; predictions cover rows 20–29 and scoring uses 22–29.
For real data, replace preparation with the existing loading → histories/ratings
→ features/labels → assembly → splits workflow. The optional environment variable
only selects a preserved source copy during independent verification.

In [1]:
from pathlib import Path
import os, sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
source_root = Path(os.environ.get("XDIYO_VERIFIED_SOURCE", str(root)))
sys.path[:0] = [str(source_root / "src"), str(source_root), str(root)]
from pathlib import Path
from functools import partial
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xdiyo_analytics.datasets import ModelDataset
from xdiyo_analytics.splits import Fold, SplitPlan
from xdiyo_analytics.analysis import PreTrainingAnalysis, PostTrainingAnalysis
from xdiyo_analytics.reporting import (
    CorrelationAnalysis, PerformanceReporter, ExperimentLeaderboardReporter,
)
from xdiyo_analytics.selection import Candidate, ModelSelection
from xdiyo_analytics.training import EstimatorAdapter, CheckpointPolicy
from xdiyo_analytics.experiments import FootballExperiment, PreparedExperiment, RefitPolicy

output = root / "experiment/football_experiment_demo/notebook"
rows = np.arange(30)
X = pd.DataFrame({"signal": np.sin(rows / 4) + rows / 20, "wave": np.cos(rows)})
y = pd.DataFrame({"total": 7 + 2 * X.signal + .1 * X.wave})
metadata = pd.DataFrame({"competition_id": 1, "season_id": 2026,
    "event_id": rows + 1001, "home_id": 1 + rows % 3, "away_id": 4 + rows % 3})
keys = ("competition_id", "season_id", "event_id")
dataset = ModelDataset(X, y, metadata, "match", keys, keys, "total")
outer = SplitPlan([Fold(rows[:20], rows[20:], rows[22:])], len(rows), rows)
inner = SplitPlan([Fold(rows[:8], rows[8:12], rows[8:12]),
                   Fold(rows[:12], rows[12:20], rows[12:20])], len(rows), rows)

def prepare():
    return PreparedExperiment(dataset, outer,
        outputs={"input_metadata": metadata.copy()}, config={"synthetic": True})

def adapter(alpha):
    return EstimatorAdapter(make_pipeline(StandardScaler(), Ridge(alpha=alpha)))

def candidate(alpha):
    return Candidate(f"Ridge {alpha}", partial(adapter, alpha),
        config={"alpha": alpha, "preprocessing": "StandardScaler"})

before = PreTrainingAnalysis({
    "Training association": CorrelationAnalysis(type="overall", partition="train"),
})
after = PostTrainingAnalysis({
    "Errors": PerformanceReporter(type="overall", partition="score", metrics=["mse", "mae"]),
    "Saved runs": ExperimentLeaderboardReporter(weights={"mse": 1.}),
})
experiment = FootballExperiment("Synthetic workflow", output_dir=output, prepare=prepare)


### 2. Fit, reuse, and reopen

The first call fits. The second call prepares the same inputs and reuses saved
numerical results. Explicit load skips preparation too. Recovered fold models
are None; predictions, labels, row identities and reports remain available.
The combined report below contains descriptive training association and final
evaluation. Rendering it does not fit or predict.

The default identity includes data/order, configuration, factories and local
library source. Use reuse=False to create a fresh execution group.

In [2]:
fixed = experiment.run(model=candidate(.1), pre_analysis=before, post_analysis=after,
                       name_fields=["alpha"])
same = experiment.run(model=candidate(.1), pre_analysis=before, post_analysis=after,
                      name_fields=["alpha"])
assert same.reused and same.record["run_id"] == fixed.record["run_id"]
assert same.training.folds[0].model is None

reopened = FootballExperiment("Synthetic workflow", output_dir=output).load(fixed.record["run_id"])
assert reopened.reused
pd.testing.assert_frame_equal(reopened.dataset.X, dataset.X)
reopened.to_html(output / "fixed.html")
print(fixed.record["name"], fixed.record["run_id"])

print("Exact reuse:", same.reused, "Loaded model:", reopened.training.folds[0].model)
reopened.to_notebook(height=850)


Ridge 0.1 · alpha=0.1 ee08c19a-2f99-4d1c-9b83-c393e3b993b6
Exact reuse: True Loaded model: None


### 3. Compare the published final runs

The leaderboard is read after publication, so it includes this final run.
Expand Run configurations to inspect its recorded settings. Internal search
trials are hidden by default. Saved native iframe interaction is verified;
live Jupyter frontend trust/display remains unverified.

In [3]:
leaderboard = experiment.leaderboard({"mse": 1.})
leaderboard.to_html(output / "leaderboard.html")
leaderboard.to_notebook(height=750)


## Continue

The [guide](../docs/analytics/football_experiment.md) adds holdout search, explicit
deployment refitting and a complete native checkpoint adapter. See the
[reference](../docs/analytics/football_experiment_reference.md),
[recovery invariants](../docs/analytics/football_experiment_equations.md),
[coverage](../docs/analytics/football_experiment_documentation_checklist.md) and
[verification record](../docs/analytics/football_experiment_check.json).
The thirteen preceding notebooks and their outputs remain unchanged.